# AETHER Kaggle T4×2 environment and model-independent acceptance

This notebook checks out immutable AETHER commit `585a63c55fa6c21a5636d9dc8b7c34d14f3f7858`, creates an isolated CPython 3.11.9 environment, installs CUDA 13.0 PyTorch, runs repository validation and all model-independent tests, executes the three-seed hierarchical graph-RL acceptance, and creates a hashed output archive.

Before running: select **GPU T4 ×2** and enable **Internet** in Kaggle settings. Keep the notebook private initially. This notebook does **not** download or run Qwen; the immutable 1.5B revision and prospective T4-FP16 qualification protocol must be frozen first.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys

WORK = Path('/kaggle/working')
# Keep large disposable installs out of Kaggle's saved output directory.
RUNTIME = Path('/tmp/aether-runtime')
RUNTIME.mkdir(parents=True, exist_ok=True)
REPO = RUNTIME / 'aether'
VENV = RUNTIME / 'aether-venv'
PY = VENV / 'bin' / 'python'
OUTPUT_ROOT = WORK / 'aether-output'
COMMIT = '585a63c55fa6c21a5636d9dc8b7c34d14f3f7858'
REPOSITORY_URL = 'https://github.com/ama1a005/aether.git'
# Kaggle's notebook backend belongs to Python 3.13 and is unavailable in the isolated 3.11 env.
os.environ['MPLBACKEND'] = 'Agg'
os.environ['PYTHONNOUSERSITE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
for gate in ('AETHER_RUN_REAL_MODEL_TESTS', 'AETHER_RUN_CALIBRATION_MODELS', 'AETHER_RUN_DEMO_E2E'):
    os.environ.pop(gate, None)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def run_logged(name, command, timeout=3600):
    log = OUTPUT_ROOT / (name + '.log')
    with log.open('w', encoding='utf-8') as handle:
        try:
            result = subprocess.run(command, cwd=REPO, env=os.environ.copy(),
                                    stdout=handle, stderr=subprocess.STDOUT,
                                    text=True, timeout=timeout, check=False)
        except subprocess.TimeoutExpired:
            print(log.read_text(encoding='utf-8')[-16000:])
            raise
    print(log.read_text(encoding='utf-8')[-16000:])
    result.check_returncode()
    return result

print('Notebook kernel:', sys.version)
print('Platform:', platform.platform())
print('Work directory:', WORK)

In [ ]:
# Fail before installation unless Kaggle supplied exactly two usable T4 GPUs.
gpu_query = subprocess.run(
    ['nvidia-smi', '--query-gpu=index,name,memory.total,memory.free',
     '--format=csv,noheader,nounits'],
    check=True, capture_output=True, text=True,
)
print(gpu_query.stdout)
gpu_rows = [row.strip() for row in gpu_query.stdout.splitlines() if row.strip()]
assert len(gpu_rows) == 2, f'Expected two GPUs, got: {gpu_rows}'
assert all('Tesla T4' in row for row in gpu_rows), gpu_rows
for row in gpu_rows:
    fields = [field.strip() for field in row.split(',')]
    assert int(fields[2]) >= 15000, f'Unexpected T4 memory: {row}'
print('KAGGLE T4x2 INVENTORY PASSED')

In [ ]:
# Install uv into the Kaggle kernel only as a Python/environment manager.
if shutil.which('uv') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
UV = shutil.which('uv')
assert UV is not None
subprocess.run([UV, '--version'], check=True)

uv_environment = os.environ.copy()
uv_environment['UV_PYTHON_INSTALL_DIR'] = str(RUNTIME / 'uv-python')
uv_environment['UV_CACHE_DIR'] = str(RUNTIME / 'uv-cache')

if not PY.is_file():
    subprocess.run(
        [UV, 'venv', '--python', '3.11.9', str(VENV)],
        check=True, env=uv_environment,
    )
assert PY.is_file(), PY
subprocess.run([str(PY), '--version'], check=True)
subprocess.run([str(PY), '-c',
                'import sys; assert sys.version_info[:3] == (3, 11, 9); print(sys.executable)'],
               check=True)

In [ ]:
# Clone once, then detach at the exact evidence commit. Existing unexpected content fails closed.
if not REPO.exists():
    subprocess.run(
        ['git', 'clone', '--branch', 'lm', '--single-branch', REPOSITORY_URL, str(REPO)],
        check=True,
    )
assert (REPO / '.git').is_dir(), f'Unexpected existing path: {REPO}'
subprocess.run(['git', 'checkout', '--detach', COMMIT], cwd=REPO, check=True)
actual_commit = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True,
).strip()
assert actual_commit == COMMIT, (actual_commit, COMMIT)
status = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True)
assert not status.strip(), status
print('Pinned clean AETHER commit:', actual_commit)

In [ ]:
# Make a Linux/CUDA installation input without altering the checked-in lock.
source_lock = REPO / 'requirements-lock.txt'
kaggle_lock = WORK / 'requirements-kaggle-base.txt'
demo_source_lock = REPO / 'requirements-demo-lock.txt'
kaggle_demo_lock = WORK / 'requirements-kaggle-demo.txt'
assert source_lock.is_file(), source_lock
lock_lines = source_lock.read_text(encoding='utf-8').splitlines()
filtered_lines = [
    line for line in lock_lines
    if line.strip() and not line.strip().startswith('torch==')
]
kaggle_lock.write_text('\n'.join(filtered_lines) + '\n', encoding='utf-8')
assert not any(line.startswith('torch==') for line in filtered_lines)
assert demo_source_lock.is_file(), demo_source_lock
demo_lines = demo_source_lock.read_text(encoding='utf-8').splitlines()
filtered_demo_lines = [
    line for line in demo_lines
    if line.strip() and not line.lstrip().startswith('#')
    and not line.strip().startswith('-r ')
    and not line.strip().lower().startswith('pywin32==')
]
kaggle_demo_lock.write_text('\n'.join(filtered_demo_lines) + '\n', encoding='utf-8')
print('Prepared:', kaggle_lock)
print('Prepared:', kaggle_demo_lock)

In [ ]:
# Seed pip because AETHER manifests invoke `python -m pip freeze --all`.
subprocess.run(
    [UV, 'pip', 'install', '--python', str(PY), 'pip==25.3'],
    check=True, env=uv_environment,
)
# Install the pinned CUDA build independently of the generic package index.
subprocess.run(
    [UV, 'pip', 'install', '--python', str(PY), 'torch==2.14.1',
     '--index-url', 'https://download.pytorch.org/whl/cu130'],
    check=True, env=uv_environment,
)
# Install the remaining research lock and AETHER without dependency re-resolution.
subprocess.run(
    [UV, 'pip', 'install', '--python', str(PY), '--requirement', str(kaggle_lock),
     '--requirement', str(kaggle_demo_lock)],
    check=True, env=uv_environment,
)
subprocess.run(
    [UV, 'pip', 'install', '--python', str(PY), '--no-deps', '--editable', str(REPO)],
    check=True, env=uv_environment,
)
subprocess.run([str(PY), '-m', 'pip', 'check'], check=True)
subprocess.run([str(PY), '-m', 'pip', '--version'], check=True)
print('PYTHON 3.11 DEPENDENCY INSTALLATION PASSED')

In [ ]:
# Verify that the isolated interpreter sees the two T4 devices and can execute FP16 kernels.
cuda_program = r'''
import sys
import torch
print('Python:', sys.version)
print('Executable:', sys.executable)
print('Torch:', torch.__version__)
print('Torch CUDA runtime:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
assert sys.version_info[:3] == (3, 11, 9)
assert torch.__version__.startswith('2.14.1')
assert torch.cuda.is_available()
assert torch.cuda.device_count() == 2
for index in range(2):
    assert 'T4' in torch.cuda.get_device_name(index)
    device = torch.device(f'cuda:{index}')
    value = torch.ones((64, 64), device=device, dtype=torch.float16)
    result = value @ value
    assert result.is_cuda and torch.isfinite(result).all()
    free, total = torch.cuda.mem_get_info(index)
    print(index, torch.cuda.get_device_name(index), free / 2**30, total / 2**30)
print('PYTHON 3.11 + CUDA T4x2 ACCEPTANCE PASSED')
'''
subprocess.run([str(PY), '-c', cuda_program], check=True)

In [ ]:
# Capture reproducibility inputs before running tests.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
environment_file = OUTPUT_ROOT / 'aether-environment.txt'
hardware_file = OUTPUT_ROOT / 'aether-hardware.txt'
commit_file = OUTPUT_ROOT / 'git_commit.txt'
freeze = subprocess.run(
    [str(PY), '-m', 'pip', 'freeze', '--all'],
    check=True, capture_output=True, text=True,
)
environment_file.write_text(freeze.stdout, encoding='utf-8')
with hardware_file.open('w', encoding='utf-8') as handle:
    subprocess.run(['nvidia-smi', '-q'], check=True, stdout=handle,
                   stderr=subprocess.STDOUT, text=True)
commit_file.write_text(COMMIT + '\n', encoding='utf-8')
print(environment_file)
print(hardware_file)
print(commit_file)

In [ ]:
# Repository policy and focused hierarchical graph-RL acceptance.
subprocess.run([str(PY), 'scripts/validate_repository.py'], cwd=REPO, check=True)
run_logged('focused-tests',
    [str(PY), '-m', 'pytest',
     'tests/unit/test_hierarchical_rl.py',
     'tests/integration/test_hierarchical_rl_acceptance.py', '-ra'],
)
print('FOCUSED HIERARCHICAL RL VERIFICATION PASSED')

In [ ]:
# Complete model-independent Python verification. Real-model gates remain skipped.
run_logged('full-tests',
    [str(PY), '-m', 'pytest', 'tests/unit', 'tests/integration', 'tests/regression', '-ra',
     '--junitxml=' + str(OUTPUT_ROOT / 'tests.xml')],
)
print('COMPLETE MODEL-INDEPENDENT SUITE PASSED')

In [ ]:
# Execute the manifest-backed three-seed branching-graph acceptance.
# Flat PPO also saves a reloadable policy; graph PPO currently saves search traces only.
run_logged('flat-ppo-training',
    [str(PY), '-m', 'experiments.phase1.rl_search_smoke',
     '--output-root', str(OUTPUT_ROOT / 'flat-ppo')])
acceptance_root = OUTPUT_ROOT / 'hierarchical-rl'
acceptance_root.mkdir(parents=True, exist_ok=True)
existing_runs = {path.resolve() for path in acceptance_root.iterdir() if path.is_dir()}
run_logged('graph-ppo-training',
    [str(PY), '-m', 'experiments.phase1.hierarchical_rl_smoke',
     '--output-root', str(acceptance_root)],
)
run_directories = [path for path in acceptance_root.iterdir()
                   if path.is_dir() and path.resolve() not in existing_runs]
assert len(run_directories) == 1, run_directories
run_directory = run_directories[0]
assert (run_directory / 'manifest.json').is_file()
assert (run_directory / 'results.json').is_file()
result = json.loads((run_directory / 'results.json').read_text(encoding='utf-8'))
assert all(result['checks'].values()), result['checks']
print('HIERARCHICAL RL ACCEPTANCE PASSED:', run_directory)

In [ ]:
# Create a single downloadable archive and independent checksum.
status = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True)
assert not status.strip(), status
shutil.copy2(kaggle_lock, OUTPUT_ROOT / kaggle_lock.name)
shutil.copy2(kaggle_demo_lock, OUTPUT_ROOT / kaggle_demo_lock.name)
file_hashes = {}
for path in sorted(OUTPUT_ROOT.rglob('*')):
    if path.is_file() and path.name != 'artifact_hashes.json':
        file_hashes[path.relative_to(OUTPUT_ROOT).as_posix()] = hashlib.sha256(path.read_bytes()).hexdigest()
(OUTPUT_ROOT / 'artifact_hashes.json').write_text(json.dumps(file_hashes, indent=2), encoding='utf-8')
archive_base = WORK / 'aether-results-585a63c'
archive = Path(shutil.make_archive(str(archive_base), 'zip', OUTPUT_ROOT))
digest = hashlib.sha256()
with archive.open('rb') as handle:
    for chunk in iter(lambda: handle.read(1024 * 1024), b''):
        digest.update(chunk)
sha256 = digest.hexdigest()
hash_file = WORK / 'aether-results-585a63c.sha256'
hash_file.write_text(f'{sha256}  {archive.name}\n', encoding='utf-8')
assert archive.stat().st_size > 0
print('Archive:', archive)
print('SHA-256:', sha256)
print('Hash file:', hash_file)

In [ ]:
# Final gate and convenient interactive download links.
from IPython.display import FileLink, display
assert actual_commit == COMMIT
assert archive.is_file() and hash_file.is_file()
print('KAGGLE MODEL-INDEPENDENT CHECKPOINT COMPLETE')
print('Qwen accessed: False')
print('Next gate: freeze immutable Qwen2.5-1.5B revision and T4-FP16 protocol')
display(FileLink(str(archive)))
display(FileLink(str(hash_file)))